# File structure thats expected:

- Single experiment folder/
  - 01
    - Labels.tif
    - CY3.5/CY5.tif
    - DAPI.tif
  - 02
  - 03
  - etc.

In [ ]:
from pathlib import Path

# --------------------------
# CONFIG
# --------------------------
FILENAME_PREFIX = "CET145"
DATASET_FOLDER  = Path('/Volumes/A4_Eva/Eva/260526_Ca_Mono_SLR1KOandWHI3KO_ASH1_HWP1/smFISH/CET145/00_Analysed')
mRNA            = "HWP1"
CY_CHANNEL      = 'CY3.5 NAR'  # change to 'CY5', 'CY3', etc.

BIN_SIZE   = 1      # µm per bin
PIXEL_SIZE = 0.065  # µm per pixel


> **Note — Imports & Configuration**
>
> Set these four variables before running anything else:
> - `FILENAME_PREFIX` — experiment identifier used in all output filenames (e.g. `"CET145"`)
> - `DATASET_FOLDER` — path to the root folder that contains numbered subfolders (`01/`, `02/`, …)
> - `CY_CHANNEL` — fluorescence channel name as it appears in the `.tif` filenames (e.g. `'CY3.5 NAR'`, `'CY5'`)
> - `PIXEL_SIZE` — microscope calibration in µm per pixel; used to convert pixel distances to µm in all outputs
>
> `BIN_SIZE` (default 1 µm) controls the spatial resolution of the intensity profiles — smaller values give finer profiles but more noise.

## Mask check

In [ ]:
# import napari

# viewer = napari.Viewer()

# KEYWORD_COLORMAP = {
#     'DAPI': 'blue',
#     'CY5': 'green'
# }

# def set_layer_style(event):
#     layer = event.value
    
#     if hasattr(layer, 'rgb') and layer.rgb:
#         layer.rgb = False
    
#     layer.blending = 'additive'
    
#     name_lower = layer.name.lower()
#     for keyword, colormap in KEYWORD_COLORMAP.items():
#         if keyword in name_lower:
#             layer.colormap = colormap
#             break

# viewer.layers.events.inserted.connect(set_layer_style)

# napari.run()

> **Note — Napari Viewer (optional, for visual QC)**
>
> Opens an interactive Napari viewer for inspecting raw images and segmentation masks before analysis. An event listener auto-applies display settings when layers are loaded:
> - Layers with `"DAPI"` in the name → blue colormap
> - Layers with `"CY5"` in the name → green colormap
> - All layers use additive blending so overlapping signals remain visible
>
> This step is purely visual and does not produce any output for the downstream pipeline. It is safe to skip if you have already verified your masks.

## Package imports

In [ ]:
from image_processing_tools.intensity_profiling import (
    # pipeline
    extract_intensity_profiles,
    # analysis
    normalize_profiles,
    flip_profile_directions,
    compute_dapi_metrics,
    compute_tip_body_ratio,
    # plotting
    plot_intensity_profiles,
    plot_dapi_heatmap,
    plot_dapi_metrics,
    plot_intensity_heatmap,
    plot_tip_body_ratio,
)
import numpy as np
import pandas as pd

> **Note — Package Imports**
>
> All analysis functions live in `image_processing_tools.intensity_profiling`:
>
> | Module | Functions |
> |---|---|
> | `skeleton.py` | `build_skeleton_graph`, `extend_skeleton` |
> | `distance.py` | `geodesic_distance_map` |
> | `extraction.py` | `extract_intensity_profiles` |
> | `analysis.py` | `normalize_profiles`, `flip_profile_directions`, `compute_dapi_metrics`, `compute_tip_body_ratio` |
> | `plotting.py` | `plot_intensity_profiles`, `plot_dapi_heatmap`, `plot_dapi_metrics`, `plot_intensity_heatmap`, `plot_tip_body_ratio` |

##  Loading from Flat Directory

> **Note — File-Reorganisation Utility *(raw cell — disabled)***
>
> This cell is stored as a **Raw** cell so it is skipped during normal execution. Run it only if all `.tif` files land in a flat directory instead of numbered subfolders.
>
> It reads the trailing number from each filename (e.g. `CET145_…_01.tif`) and moves the file into a matching subfolder (`01/`, `02/`, …), creating the directory layout expected by the extraction loop.

In [ ]:
df, df_dapi = extract_intensity_profiles(
    DATASET_FOLDER, FILENAME_PREFIX, CY_CHANNEL, PIXEL_SIZE, BIN_SIZE
)

> **Note — Data Extraction**
>
> `extract_intensity_profiles` runs the full pipeline for every numbered subfolder:
> skeletonise → extend endpoints → locate tip via DAPI centroid → compute geodesic
> distances → bin mean intensities. Returns two DataFrames:
> - `df` — fluorescence (CY channel) intensity vs. distance from tip
> - `df_dapi` — DAPI intensity vs. distance from tip
>
> Both have columns: `label` (`"{subfolder}_{label_id}"`), `distance` (µm), `intensity`.

In [ ]:
print(df['label'].unique())
print(len(df))

> **Note — Label Verification**
>
> Prints all unique hypha labels (format `"{subfolder}_{label_id}"`, e.g. `"02_5"`) and total row count. Use this to confirm the expected number of hyphae were extracted before continuing to plotting.

## Loading CY Channel Intensity Profiles from csv

> **Note — Alternative Entry Point: Load from Pre-computed CSVs *(raw cells — disabled)***
>
> Cells 11–13 are stored as Raw cells and are skipped during normal execution. They provide an alternative starting point if you want to load previously saved CSV files instead of re-running the extraction loop.
>
> To use: convert the relevant raw cell to a code cell and update the `glob` pattern to match your CSV paths. Each CSV must have `label`, `distance`, and `intensity` columns matching the format produced by the extraction loop above.

## Raw Intensity Profiles

In [ ]:
plot_intensity_profiles(
    df, mRNA, FILENAME_PREFIX, DATASET_FOLDER,
    xlim=(0, 40), ylim=(0, 17500),
)


> **Note — Raw Fluorescence Intensity Profiles**
>
> Plots one semi-transparent line per hypha showing mean raw fluorescence intensity vs. geodesic distance from the tip. No averaging across hyphae is applied (`estimator=None`).
>
> Adjust `plt.xlim` to match the length range of your hyphae, and `plt.ylim` / `plt.yticks` to fit the raw intensity range of your fluorophore/exposure. Saved as PDF (vector) and PNG (300 dpi).

# Normalize data based on highest intensity per hyphae

In [ ]:
df_norm = normalize_profiles(df)

> **Note — Per-Hypha Normalization**
>
> Divides each hypha's intensity values by its own maximum, rescaling all profiles to [0, 1]. This removes cell-to-cell variation in absolute brightness (expression level, z-position, bleaching) and makes the spatial *pattern* of localization directly comparable across cells.
>
> The normalized data is stored in `df_norm`; the original `df` remains unmodified and is used later for the tip:body ratio, which requires absolute intensities.

In [ ]:
plot_intensity_profiles(
    df_norm, mRNA, FILENAME_PREFIX, DATASET_FOLDER,
    xlim=(0, 40), normalized=True,
)

> **Note — Normalized Intensity Profiles**
>
> Same spaghetti-line plot as the raw version but using `df_norm` (y-axis 0–1). Better for comparing the localization pattern shape across hyphae with different absolute expression levels.

## DAPI Sanity Check

In [ ]:
plot_dapi_heatmap(df_dapi, FILENAME_PREFIX, DATASET_FOLDER, bin_size=BIN_SIZE)

> **Note — DAPI Sanity-Check Heatmap**
>
> A 2-D heatmap of DAPI intensity (blue) sorted by hypha length. Because DAPI labels nuclear DNA, signal should be concentrated at **large distances from the tip** (i.e. near the yeast body/base), not near 0.
>
> This plot is a **quality-control step**: any hypha showing DAPI peak near distance = 0 likely has a reversed direction assignment and needs to be added to `FLIP_LABELS`.

## DAPI METRICS

In [ ]:
# Set to a number (e.g. 500) to override automatic Otsu threshold
DAPI_THRESHOLD_OVERRIDE = None

df_dapi_metrics, dapi_threshold = compute_dapi_metrics(
    df_dapi, dapi_threshold=DAPI_THRESHOLD_OVERRIDE
)
df_dapi_metrics['strain'] = FILENAME_PREFIX
df_dapi_metrics.to_csv(
    DATASET_FOLDER / f'{FILENAME_PREFIX}_dapi_metrics.csv', index=False
)
plot_dapi_metrics(df_dapi_metrics, FILENAME_PREFIX, DATASET_FOLDER, dapi_threshold)


> **Note — DAPI Metrics: Spread and Centroid Position**
>
> Quantifies nuclear position along each hypha with two scalar metrics:
>
> - **DAPI spread** — span (µm) of the hypha where DAPI signal exceeds the Otsu threshold; reflects how extended the nucleus is.
> - **DAPI centroid** — intensity-weighted mean distance from the tip; larger values mean the nucleus is further from the tip (closer to the yeast body).
>
> The Otsu threshold is computed automatically unless `DAPI_THRESHOLD_OVERRIDE` is set. Results are saved to `{FILENAME_PREFIX}_dapi_metrics.csv` and plotted as a jittered scatter with the median marked in red.

## Manual Direction Corrections (Only run when needed)

In [ ]:
# Add labels that appear reversed in the DAPI heatmap or profile plots.
# Format: "{subfolder}_{label_id}"
FLIP_LABELS = [
    '04_3', '01_3', '01_1', '03_6', '06_7', '04_4',
    '01_2', '02_1', '03_4', '04_1', '02_5', '03_7',
    '01_6', '06_5', '06_1',
]

df_norm = flip_profile_directions(df_norm, FLIP_LABELS)
df_dapi = flip_profile_directions(df_dapi, FLIP_LABELS)


> **Note — Manual Direction Corrections (second, more complete pass)**
>
> A second flip pass applied *after* normalization. `FLIP_LABELS` was assembled by cross-referencing the DAPI sanity heatmap and the profile plots to catch all remaining reversed hyphae.
>
> This block updates `df_norm` and `df_dapi` (but not the raw `df`). Run it before generating the final heatmaps.

## Intensity Heatmap


In [ ]:
plot_intensity_heatmap(
    df_norm, mRNA, FILENAME_PREFIX, DATASET_FOLDER,
    bin_size=BIN_SIZE, xlim=(0, 40), vmax=1,
)


> **Note — Normalized Intensity Heatmap (main output)**
>
> The primary summary figure. A 2-D matrix where:
> - **Rows** = individual hyphae sorted from shortest (top) to longest (bottom)
> - **Columns** = distance bins from tip (µm), x-axis limited to 40 µm
> - **Colour** = normalized mRNA fluorescence intensity (plasma colormap, `VMAX = 1`)
>
> Hyphae shorter than the full x-range will be NaN (missing/white) beyond their length, creating the characteristic diagonal boundary when sorted by length. Saved to a dedicated `heatmaps/` subfolder.

## Plot sanity check

In [ ]:
# Coarser bins + hypha ID labels — use to spot and name reversed profiles
plot_intensity_heatmap(
    df_norm, mRNA, FILENAME_PREFIX, DATASET_FOLDER,
    bin_size=2, xlim=(0, 40), vmax=1, show_labels=True,
)
df_norm.to_csv(
    DATASET_FOLDER / f'{FILENAME_PREFIX}_intensity_profiles.csv', index=False
)


> **Note — Sanity-Check Heatmap with Hypha ID Labels**
>
> An annotated version of the heatmap above with hypha ID labels on the y-axis (e.g. `"02_5"`). Uses a coarser 2 µm bin size to keep the matrix manageable.
>
> Use this to pinpoint *which specific hyphae* have unusual profiles by matching the row position to its label. Also saves `df_norm` to `{FILENAME_PREFIX}_intensity_profiles.csv` for downstream analysis.

## Tip to body ratio

- Ratio = 1 — equal intensity in both zones, no localization
- Ratio > 1 — tip brighter than body = localized
- Ratio < 1 — body brighter than tip

In [ ]:
TIP_ZONE  = (0, 5)   # µm
BODY_ZONE = (5, 10)  # µm — same width as tip zone

df_ratio = compute_tip_body_ratio(df, tip_zone=TIP_ZONE, body_zone=BODY_ZONE)
df_ratio['strain'] = FILENAME_PREFIX
df_ratio.to_csv(
    DATASET_FOLDER / f'{FILENAME_PREFIX}_tip_body_ratio.csv', index=False
)
plot_tip_body_ratio(
    df_ratio, FILENAME_PREFIX, DATASET_FOLDER,
    tip_zone=TIP_ZONE, body_zone=BODY_ZONE,
)


> **Note — Tip:Body Intensity Ratio**
>
> Computes a single summary statistic per hypha: the ratio of mean fluorescence in the **tip zone** (0–5 µm from tip) to the **body zone** (5–10 µm from tip). Hyphae shorter than `BODY_ZONE_END` are excluded.
>
> Calculated on the raw (unnormalized) `df` so that absolute intensity differences between zones are captured. A ratio > 1 indicates tip-enriched mRNA localization. Saved to `{FILENAME_PREFIX}_tip_body_ratio.csv` and plotted as a jittered strip chart with median and a reference line at ratio = 1.